# 07 — Training Without Color Jitter

Ablation of the color augmentation. The flat, stage-1, and stage-2 models are retrained for both architectures and all three seeds without ColorJitter; every other setting, including the unweighted loss and the seeds, is identical to notebook 01. The main models from notebooks 01–02 are reused for comparison and remain the main results. All settings and analyses below were fixed before training, and the results are reported regardless of their direction.

## 1. Configuration

In [1]:
import torch
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

# Paths (adjust to your Google Drive)
DATA_DIR  = Path('/content/drive/MyDrive/USK-Coffee')
LOCAL_DIR = Path('/content/USK-Coffee')
OUT_ROOT  = Path('/content/drive/MyDrive/coffee-two-stage/models')                  # models from notebook 01
NJ_ROOT   = Path('/content/drive/MyDrive/coffee-two-stage/models_no_color_jitter')  # new models without color jitter
TEST_OUT  = Path('/content/drive/MyDrive/coffee-two-stage/test_results_no_color_jitter')

# Data
CLASSES = ['defect', 'longberry', 'peaberry', 'premium']
EXPECTED_COUNT = {'train': 1200, 'val': 400, 'test': 400}
IMG_SIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# Training (same as notebook 01)
BATCH_SIZE = 32
LR = 1e-4
STEP_SIZE, GAMMA = 5, 0.5
MAX_EPOCHS = 30
PATIENCE = 5
NUM_WORKERS = 2

ARCHS = ['resnet50', 'efficientnet_b0']
SEEDS = [0, 1, 2]
ALPHA = 0.05

# Same label schemes as notebook 01
# Stage 1: 0 defect, 1 longberry, 2 peaberry+premium; stage 2: 0 peaberry, 1 premium
SCHEMES_MAP = {
    '4class': {'defect': 0, 'longberry': 1, 'peaberry': 2, 'premium': 3},
    '3class': {'defect': 0, 'longberry': 1, 'peaberry': 2, 'premium': 2},
    '2class': {'peaberry': 0, 'premium': 1},
}
NUM_CLASSES = {'4class': 4, '3class': 3, '2class': 2}
SCHEMES = ['4class', '3class', '2class']

def run_dir(arch, scheme, seed):
    return OUT_ROOT / arch / scheme / f'seed{seed}'

def nj_dir(arch, scheme, seed):
    return NJ_ROOT / arch / scheme / f'seed{seed}'

TEST_OUT.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


## 2. Copy data to the Colab disk

In [2]:
import shutil, time

EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

def list_images(folder):
    return sorted(f for f in folder.iterdir() if f.suffix.lower() in EXTS)

t0 = time.time()
for split in ['train', 'val', 'test']:
    dst = LOCAL_DIR / split
    ok = dst.exists() and all(
        (dst / c).exists() and len(list_images(dst / c)) == EXPECTED_COUNT[split]
        for c in CLASSES)
    if not ok:
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(DATA_DIR / split, dst)
    for c in CLASSES:
        assert len(list_images(dst / c)) == EXPECTED_COUNT[split], f'{split}/{c} incomplete'
print(f'done in {time.time() - t0:.0f}s')

done in 120s


## 3. Dataset and transforms

In [3]:
import random
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Same augmentation as notebook 01 except that ColorJitter is removed
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class CoffeeDataset(Dataset):
    def __init__(self, split_dir, scheme, transform):
        self.samples = [(f, label) for c, label in SCHEMES_MAP[scheme].items()
                        for f in list_images(split_dir / c)]
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        with Image.open(path) as img:
            return self.transform(img.convert('RGB')), label

## 4. Training function

In [4]:
import pandas as pd
import torch.nn as nn
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(_):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

def build_model(arch, num_classes, pretrained=True):
    if arch == 'resnet50':
        m = models.resnet50(weights='IMAGENET1K_V1' if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif arch == 'efficientnet_b0':
        m = models.efficientnet_b0(weights='IMAGENET1K_V1' if pretrained else None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    else:
        raise ValueError(arch)
    return m

@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs, labels = [], []
    for x, y in loader:
        probs.append(torch.softmax(model(x.to(device)), 1).cpu())
        labels.append(y)
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()

def epochs_since_improvement(history):
    losses = [h['val_loss'] for h in history]
    return len(losses) - 1 - int(np.argmin(losses))

def should_stop(history):
    return len(history) >= MAX_EPOCHS or (len(history) > 0 and epochs_since_improvement(history) >= PATIENCE)

# Identical to train_model in notebook 01 (unweighted loss); only train_tf differs
def train_model(arch, scheme, seed, out_dir, pretrained=True):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path  = out_dir / 'last_checkpoint.pt'
    best_path  = out_dir / 'best_model.pt'
    final_path = out_dir / 'final_model.pt'
    if final_path.exists():
        print(f'skip (done): {out_dir}')
        return

    set_seed(seed)
    train_ds = CoffeeDataset(LOCAL_DIR / 'train', scheme, train_tf)
    val_ds   = CoffeeDataset(LOCAL_DIR / 'val', scheme, eval_tf)
    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                              worker_init_fn=seed_worker, generator=g, pin_memory=True)
    val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    model = build_model(arch, NUM_CLASSES[scheme], pretrained).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=GAMMA)

    history = []
    # Resume after a Colab disconnect
    if ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer'])
        scheduler.load_state_dict(ck['scheduler']); history = ck['history']
        g.set_state(ck['rng_gen']); torch.set_rng_state(ck['rng_torch'])
        np.random.set_state(ck['rng_np']); random.setstate(ck['rng_py'])
        if ck['rng_cuda'] is not None: torch.cuda.set_rng_state_all(ck['rng_cuda'])
        print(f'resuming from epoch {len(history) + 1}')

    while not should_stop(history):
        epoch = len(history) + 1
        t0 = time.time()
        lr = optimizer.param_groups[0]['lr']
        model.train()
        loss_sum, correct, total = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item(); total += len(y)
        scheduler.step()

        prob, y_val = predict(model, val_loader)
        val_loss = nn.functional.nll_loss(torch.log(torch.tensor(prob) + 1e-12), torch.tensor(y_val)).item()
        row = {'epoch': epoch, 'lr': lr, 'train_loss': loss_sum / total, 'train_acc': correct / total,
               'val_loss': val_loss, 'val_acc': float((prob.argmax(1) == y_val).mean()), 'sec': time.time() - t0}

        # Keep the weights with the lowest validation loss
        if row['val_loss'] < min((h['val_loss'] for h in history), default=float('inf')):
            torch.save(model.state_dict(), best_path)
        history.append(row)

        torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(), 'history': history,
                    'rng_gen': g.get_state(), 'rng_torch': torch.get_rng_state(),
                    'rng_np': np.random.get_state(), 'rng_py': random.getstate(),
                    'rng_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None},
                   ckpt_path)
        pd.DataFrame(history).to_csv(out_dir / 'history.csv', index=False)
        print(f"  epoch {epoch:>2}/{MAX_EPOCHS} | train {row['train_loss']:.4f}/{row['train_acc']:.4f} "
              f"| val {val_loss:.4f}/{row['val_acc']:.4f} | {row['sec']:.0f}s")

    hist = pd.DataFrame(history)
    best = hist.loc[hist['val_loss'].idxmin()]
    print(f"  stopped after epoch {len(hist)} | best val_loss {best['val_loss']:.4f} "
          f"(val_acc {best['val_acc']:.4f}) at epoch {int(best['epoch'])}")

    model.load_state_dict(torch.load(best_path, map_location=device))
    full_val = CoffeeDataset(LOCAL_DIR / 'val', '4class', eval_tf)
    prob, y_true = predict(model, DataLoader(full_val, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS))
    np.savez(out_dir / 'val_predictions.npz', prob=prob, y_true=y_true,
             paths=np.array([str(p.relative_to(LOCAL_DIR)) for p, _ in full_val.samples]))

    best_path.replace(final_path)
    ckpt_path.unlink(missing_ok=True)

## 5. Train the 18 models

In [5]:
# Completed runs are skipped, interrupted runs resume
runs = [(seed, arch, scheme) for seed in SEEDS for arch in ARCHS for scheme in SCHEMES]

for i, (seed, arch, scheme) in enumerate(runs, 1):
    print(f'[{i:>2}/{len(runs)}] {arch} | {scheme} | seed {seed}')
    train_model(arch, scheme, seed, nj_dir(arch, scheme, seed))
print('All runs finished')

[ 1/18] resnet50 | 4class | seed 0
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 159MB/s]


  epoch  1/30 | train 0.4011/0.8515 | val 0.3154/0.8838 | 56s
  epoch  2/30 | train 0.1628/0.9406 | val 0.2758/0.9081 | 57s
  epoch  3/30 | train 0.1149/0.9573 | val 0.1797/0.9425 | 60s
  epoch  4/30 | train 0.1063/0.9642 | val 0.1935/0.9313 | 59s
  epoch  5/30 | train 0.0755/0.9735 | val 0.2543/0.9106 | 59s
  epoch  6/30 | train 0.0405/0.9873 | val 0.1020/0.9637 | 59s
  epoch  7/30 | train 0.0277/0.9890 | val 0.1349/0.9525 | 60s
  epoch  8/30 | train 0.0291/0.9917 | val 0.1602/0.9431 | 60s
  epoch  9/30 | train 0.0214/0.9940 | val 0.1125/0.9656 | 60s
  epoch 10/30 | train 0.0228/0.9933 | val 0.2012/0.9325 | 61s
  epoch 11/30 | train 0.0193/0.9927 | val 0.0955/0.9656 | 60s
  epoch 12/30 | train 0.0134/0.9967 | val 0.1485/0.9513 | 61s
  epoch 13/30 | train 0.0093/0.9977 | val 0.1198/0.9606 | 61s
  epoch 14/30 | train 0.0090/0.9979 | val 0.0919/0.9669 | 61s
  epoch 15/30 | train 0.0075/0.9983 | val 0.1369/0.9550 | 60s
  epoch 16/30 | train 0.0072/0.9985 | val 0.1156/0.9681 | 61s
  epoch 

100%|██████████| 20.5M/20.5M [00:00<00:00, 140MB/s] 


  epoch  1/30 | train 0.6669/0.7646 | val 0.5094/0.8263 | 41s
  epoch  2/30 | train 0.2047/0.9296 | val 0.2640/0.9100 | 33s
  epoch  3/30 | train 0.1318/0.9529 | val 0.3504/0.8825 | 36s
  epoch  4/30 | train 0.1110/0.9610 | val 0.2002/0.9306 | 34s
  epoch  5/30 | train 0.0826/0.9735 | val 0.1970/0.9281 | 35s
  epoch  6/30 | train 0.0709/0.9767 | val 0.1490/0.9487 | 37s
  epoch  7/30 | train 0.0524/0.9806 | val 0.1626/0.9425 | 34s
  epoch  8/30 | train 0.0409/0.9852 | val 0.1737/0.9419 | 34s
  epoch  9/30 | train 0.0421/0.9858 | val 0.1939/0.9387 | 35s
  epoch 10/30 | train 0.0404/0.9854 | val 0.1412/0.9544 | 35s
  epoch 11/30 | train 0.0356/0.9875 | val 0.1382/0.9525 | 35s
  epoch 12/30 | train 0.0255/0.9919 | val 0.1880/0.9406 | 34s
  epoch 13/30 | train 0.0244/0.9925 | val 0.1467/0.9494 | 36s
  epoch 14/30 | train 0.0285/0.9908 | val 0.1825/0.9419 | 34s
  epoch 15/30 | train 0.0263/0.9917 | val 0.1641/0.9425 | 35s
  epoch 16/30 | train 0.0254/0.9910 | val 0.1647/0.9437 | 37s
  stoppe

## 6. Training status

In [6]:
status = []
for seed, arch, scheme in runs:
    d = nj_dir(arch, scheme, seed)
    row = {'arch': arch, 'scheme': scheme, 'seed': seed}
    if (d / 'history.csv').exists():
        h = pd.read_csv(d / 'history.csv')
        b = h.loc[h['val_loss'].idxmin()]
        row.update({'epochs_run': len(h), 'best_epoch': int(b['epoch']),
                    'best_val_loss': round(b['val_loss'], 4), 'val_acc_at_best': round(b['val_acc'], 4),
                    'train_minutes': round(h['sec'].sum() / 60, 1)})
    row['status'] = ('done' if (d / 'final_model.pt').exists()
                     else 'running/interrupted' if (d / 'history.csv').exists() else 'not started')
    status.append(row)
status = pd.DataFrame(status)
status.to_csv(TEST_OUT / 'training_status_nojitter.csv', index=False)
print(status.to_string(index=False))

           arch scheme  seed  epochs_run  best_epoch  best_val_loss  val_acc_at_best  train_minutes status
       resnet50 4class     0          19          14         0.0919           0.9669           18.9   done
       resnet50 3class     0          11           6         0.1205           0.9550           10.9   done
       resnet50 2class     0          13           8         0.0466           0.9875            6.7   done
efficientnet_b0 4class     0          16          11         0.1382           0.9525            9.4   done
efficientnet_b0 3class     0          12           7         0.1580           0.9531            7.0   done
efficientnet_b0 2class     0          22          17         0.0455           0.9850            6.8   done
       resnet50 4class     1          11           6         0.1031           0.9694           11.1   done
       resnet50 3class     1          16          11         0.1166           0.9600           15.8   done
       resnet50 2class     1         

## 7. Test predictions

In [7]:
test_ds = CoffeeDataset(LOCAL_DIR / 'test', '4class', eval_tf)
test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_paths = np.array([str(p.relative_to(LOCAL_DIR)) for p, _ in test_ds.samples])

for seed, arch, scheme in runs:
    d = nj_dir(arch, scheme, seed)
    f = d / 'test_predictions.npz'
    if f.exists():
        print(f'skip (already predicted): {d}')
        continue
    model = build_model(arch, NUM_CLASSES[scheme], pretrained=False)
    model.load_state_dict(torch.load(d / 'final_model.pt', map_location='cpu'))
    prob, y_true = predict(model.to(device), test_loader)
    np.savez(f, prob=prob, y_true=y_true, paths=test_paths)
    print(f'predicted: {d}')

## 8. Metrics per seed

In [8]:
from sklearn.metrics import precision_recall_fscore_support

def load_npz(path):
    z = np.load(path)
    assert (z['paths'] == test_paths).all(), f'image order mismatch: {path}'
    return z

# Stage 1 = 0 defect, 1 longberry, 2 peaberry+premium; stage 2 = 0 peaberry, 1 premium
def two_stage_predict(s1_prob, s2_prob):
    s1, s2 = s1_prob.argmax(1), s2_prob.argmax(1)
    return np.where(s1 == 2, 2 + s2, s1)

def pair_errors(y, pred, a, b):
    return int(((y == a) & (pred == b)).sum() + ((y == b) & (pred == a)).sum())

def metrics_row(y, pred):
    p, r, f1, _ = precision_recall_fscore_support(y, pred, labels=range(4), zero_division=0)
    row = {'accuracy': (y == pred).mean(), 'macro_f1': f1.mean(), 'errors': int((y != pred).sum()),
           'pb_pr_errors': pair_errors(y, pred, 2, 3)}
    for k, c in enumerate(CLASSES):
        row.update({f'P_{c}': p[k], f'R_{c}': r[k], f'F1_{c}': f1[k]})
    return row

def stage_details(y, s1_prob, s2_prob):
    s1_pred = s1_prob.argmax(1)
    pair = np.isin(y, [2, 3])
    s1_true = np.where(pair, 2, y)
    s2_true = (y == 3).astype(int)
    return {'stage1_acc': (s1_true == s1_pred).mean(),
            'stage2_acc_on_pair': (s2_prob.argmax(1)[pair] == s2_true[pair]).mean(),
            'pair_lost_at_stage1': int((pair & (s1_pred != 2)).sum()),
            'other_routed_to_stage2': int((~pair & (s1_pred == 2)).sum()),
            'routed_to_stage2_share': (s1_pred == 2).mean()}

rows, preds = [], {}
for arch in ARCHS:
    for seed in SEEDS:
        z = {('main', s): load_npz(run_dir(arch, s, seed) / 'test_predictions.npz') for s in SCHEMES}
        z.update({('nj', s): load_npz(nj_dir(arch, s, seed) / 'test_predictions.npz') for s in SCHEMES})
        y = z[('main', '4class')]['y_true']
        p = {'flat': z[('main', '4class')]['prob'].argmax(1),
             'hierarchical': two_stage_predict(z[('main', '3class')]['prob'], z[('main', '2class')]['prob']),
             'flat_nojitter': z[('nj', '4class')]['prob'].argmax(1),
             'hierarchical_nojitter': two_stage_predict(z[('nj', '3class')]['prob'], z[('nj', '2class')]['prob'])}
        preds[(arch, seed)] = (y, p)
        for name, pred in p.items():
            row = {'arch': arch, 'seed': seed, 'model': name, **metrics_row(y, pred)}
            if name.startswith('hierarchical'):
                key = 'main' if name == 'hierarchical' else 'nj'
                row.update(stage_details(y, z[(key, '3class')]['prob'], z[(key, '2class')]['prob']))
            rows.append(row)

results = pd.DataFrame(rows)
results.to_csv(TEST_OUT / 'test_results_per_seed_nojitter.csv', index=False)

main_cols = ['accuracy', 'macro_f1', 'pb_pr_errors', 'errors', 'R_peaberry', 'P_peaberry', 'R_premium', 'P_premium']
stage_cols = ['stage1_acc', 'stage2_acc_on_pair', 'pair_lost_at_stage1',
              'other_routed_to_stage2', 'routed_to_stage2_share']
pd.set_option('display.width', 220)
print(results[['arch', 'model', 'seed'] + main_cols].round(4).to_string(index=False))
print('\nStage details:')
print(results[results['model'].str.startswith('hierarchical')][['arch', 'model', 'seed'] + stage_cols]
      .round(4).to_string(index=False))

           arch                 model  seed  accuracy  macro_f1  pb_pr_errors  errors  R_peaberry  P_peaberry  R_premium  P_premium
       resnet50                  flat     0    0.8019    0.8002           112     317      0.7125      0.9283     0.9300     0.6584
       resnet50          hierarchical     0    0.8162    0.8201            83     294      0.8025      0.8966     0.9050     0.6523
       resnet50         flat_nojitter     0    0.8462    0.8492            93     246      0.7850      0.9155     0.9325     0.6907
       resnet50 hierarchical_nojitter     0    0.8425    0.8436           114     252      0.7300      0.9012     0.9150     0.7079
       resnet50                  flat     1    0.7950    0.8002            93     328      0.7825      0.8719     0.9575     0.6060
       resnet50          hierarchical     1    0.8662    0.8665            41     214      0.9275      0.8771     0.9100     0.7745
       resnet50         flat_nojitter     1    0.8812    0.8817            5

## 9. Mean and standard deviation

In [9]:
summary = results.groupby(['arch', 'model'], sort=False)[main_cols + stage_cols].agg(['mean', 'std'])
summary.to_csv(TEST_OUT / 'test_results_summary_nojitter.csv')
print(summary.round(4).to_string())

                                      accuracy         macro_f1         pb_pr_errors             errors          R_peaberry         P_peaberry         R_premium         P_premium         stage1_acc         stage2_acc_on_pair         pair_lost_at_stage1          other_routed_to_stage2          routed_to_stage2_share        
                                          mean     std     mean     std         mean      std      mean      std       mean     std       mean     std      mean     std      mean     std       mean     std               mean     std                mean      std                   mean      std                   mean     std
arch            model                                                                                                                                                                                                                                                                                                               
resnet50        flat     

## 10. McNemar test (two-stage vs flat, both without color jitter)

In [10]:
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

# Same procedure as notebook 02
tests = []
for arch in ARCHS:
    for seed in SEEDS:
        y, p = preds[(arch, seed)]
        pf, ph = p['flat_nojitter'], p['hierarchical_nojitter']
        for subset, mask in [('all_1600', np.ones_like(y, bool)), ('pb_pr_800', np.isin(y, [2, 3]))]:
            cf, ch = (pf == y)[mask], (ph == y)[mask]
            b = int((cf & ~ch).sum())   # correct by flat only
            c = int((~cf & ch).sum())   # correct by two-stage only
            table = [[int((cf & ch).sum()), b], [c, int((~cf & ~ch).sum())]]
            direction = 'hier_better' if c > b else ('flat_better' if b > c else 'equal')
            tests.append({'arch': arch, 'seed': seed, 'subset': subset,
                          'acc_flat': cf.mean(), 'acc_hier': ch.mean(),
                          'b_flat_only': b, 'c_hier_only': c, 'direction': direction,
                          'p_exact': mcnemar(table, exact=True).pvalue})

mc = pd.DataFrame(tests)
mc['p_holm'] = np.nan
for arch in ARCHS:
    idx = mc['arch'] == arch
    mc.loc[idx, 'p_holm'] = multipletests(mc.loc[idx, 'p_exact'], alpha=ALPHA, method='holm')[1]
mc['significant'] = mc['p_holm'] < ALPHA
mc.to_csv(TEST_OUT / 'mcnemar_results_nojitter.csv', index=False)
print(mc.round(4).to_string(index=False))

# Consistency criterion: significant in all three seeds in the same direction
rows = []
for (arch, subset), g in mc.groupby(['arch', 'subset'], sort=False):
    n_hier = int((g['significant'] & (g['direction'] == 'hier_better')).sum())
    n_flat = int((g['significant'] & (g['direction'] == 'flat_better')).sum())
    if n_hier == len(SEEDS):
        verdict = 'consistent: two-stage better'
    elif n_flat == len(SEEDS):
        verdict = 'consistent: flat better'
    elif n_hier + n_flat > 0:
        verdict = 'inconsistent across seeds'
    else:
        verdict = 'no significant difference'
    rows.append({'arch': arch, 'subset': subset, 'sig_hier_better': f'{n_hier}/{len(SEEDS)}',
                 'sig_flat_better': f'{n_flat}/{len(SEEDS)}', 'verdict': verdict})
conclusion = pd.DataFrame(rows)
conclusion.to_csv(TEST_OUT / 'mcnemar_conclusion_nojitter.csv', index=False)
print(conclusion.to_string(index=False))

           arch  seed    subset  acc_flat  acc_hier  b_flat_only  c_hier_only   direction  p_exact  p_holm  significant
       resnet50     0  all_1600    0.8462    0.8425           94           88 flat_better   0.7110  1.0000        False
       resnet50     0 pb_pr_800    0.8588    0.8225           53           24 flat_better   0.0013  0.0063         True
       resnet50     1  all_1600    0.8812    0.8412          150           86 flat_better   0.0000  0.0002         True
       resnet50     1 pb_pr_800    0.8888    0.8725           51           38 flat_better   0.2031  0.7214        False
       resnet50     2  all_1600    0.8219    0.8088          122          101 flat_better   0.1803  0.7214        False
       resnet50     2 pb_pr_800    0.8038    0.8000           50           47 flat_better   0.8392  1.0000        False
efficientnet_b0     0  all_1600    0.8906    0.8606           88           40 flat_better   0.0000  0.0001         True
efficientnet_b0     0 pb_pr_800    0.926

## 11. Mean difference across seeds with 95% CI

In [11]:
from scipy import stats

# Same as the manuscript: Student's t, df = number of seeds - 1
t_crit = stats.t.ppf(0.975, len(SEEDS) - 1)
comparisons = [('hierarchical_nojitter', 'flat_nojitter'),   # two-stage vs flat without color jitter
               ('flat_nojitter', 'flat'),                     # effect of removing color jitter
               ('hierarchical_nojitter', 'hierarchical')]
ci_rows = []
for arch in ARCHS:
    for model_a, model_b in comparisons:
        for subset, cls in [('all_1600', None), ('pb_pr_800', [2, 3])]:
            diffs = []
            for seed in SEEDS:
                y, p = preds[(arch, seed)]
                mask = np.ones_like(y, bool) if cls is None else np.isin(y, cls)
                diffs.append(100 * ((p[model_a] == y)[mask].mean() - (p[model_b] == y)[mask].mean()))
            diffs = np.array(diffs)
            half = t_crit * diffs.std(ddof=1) / np.sqrt(len(diffs))
            ci_rows.append({'arch': arch, 'comparison': f'{model_a} - {model_b}', 'subset': subset,
                            **{f'diff_seed{s}': round(d, 2) for s, d in zip(SEEDS, diffs)},
                            'mean_diff': round(diffs.mean(), 2),
                            'ci_low': round(diffs.mean() - half, 2), 'ci_high': round(diffs.mean() + half, 2)})
ci = pd.DataFrame(ci_rows)
ci.to_csv(TEST_OUT / 'ci_difference_nojitter.csv', index=False)
print(ci.to_string(index=False))

           arch                            comparison    subset  diff_seed0  diff_seed1  diff_seed2  mean_diff  ci_low  ci_high
       resnet50 hierarchical_nojitter - flat_nojitter  all_1600       -0.37       -4.00       -1.31      -1.90   -6.57     2.78
       resnet50 hierarchical_nojitter - flat_nojitter pb_pr_800       -3.63       -1.62       -0.37      -1.87   -5.95     2.20
       resnet50                  flat_nojitter - flat  all_1600        4.44        8.62       -5.06       2.67  -14.76    20.09
       resnet50                  flat_nojitter - flat pb_pr_800        3.75        1.88       -8.88      -1.08  -18.01    15.84
       resnet50  hierarchical_nojitter - hierarchical  all_1600        2.62       -2.50       -1.94      -0.60   -7.59     6.38
       resnet50  hierarchical_nojitter - hierarchical pb_pr_800       -3.12       -4.62       -8.75      -5.50  -12.74     1.74
efficientnet_b0 hierarchical_nojitter - flat_nojitter  all_1600       -3.00       -2.37        1.62     